In [ ]:
import os
import json
import chromadb
import cohere
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv


ModuleNotFoundError: No module named 'openai'

In [5]:
load_dotenv()

True

In [6]:
# ....
# same code in retrieval_augmented_generation/basic_rag_architecture_and_implementation.ipynb
# ....

In [ ]:
# veryfy step
python -c "import gitquest; result = gitquest.ask_gitquest('how do I create a new branch'); print(result['answer'])"

In [ ]:
"""
Since we're aiming for the modern recommended approach, we'd like retrieval to
surface git restore --staged <file>, the command Git documentation now
steers users toward for removing a file from the staging area while keeping
working directory changes intact.

manually searching corpus.jsonl, this command is seen to have a chunk_id of d5237eadab2f87ed
and title git-restore(1) :: NAME (part 4).
"""

chunks = retrieve("how do I unstage a file I accidentally added?", n_results=20)
for i, chunk in enumerate(chunks, start=1):
    print(f"  [{i}] {chunk['distance']:.4f}  {chunk['title']}")

In [ ]:
# NOTE
"""
A fundamental characteristic of embedding-based retrieval.
Measures proximity between the words that are present,
not conceptual equivalence between what a user has in mind and
the terminology a documentation author chose.


Look at how we can use the LLM itself to generate alternative phrasings
of a query and run multiple retrievals,
giving us a much better shot at surfacing the right chunks.

idea is to use the LLM
"""

'\nA fundamental characteristic of embedding-based retrieval.\nMeasures proximity between the words that are present,\nnot conceptual equivalence between what a user has in mind and\nthe terminology a documentation author chose.\n\n\nLook at how we can use the LLM itself to generate alternative phrasings\nof a query and run multiple retrievals,\ngiving us a much better shot at surfacing the right chunks.\n'

In [8]:
MAX_REFORMULATIONS = 3

EXPANSION_PROMPT = """You are helping improve search over Git documentation.

Given a user's question, generate 2 or 3 alternative phrasings that use different vocabulary but ask the same thing. Use terminology that might appear in official Git documentation (command names, flags, technical terms).

Return ONLY the alternative phrasings, one per line, with no numbering, no bullet points, no explanation, and no blank lines.

User question: {query}"""

In [ ]:
def expand_query(query):
    response = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": EXPANSION_PROMPT.format(query=query)}],
        # use temperature=0.3 to keep the reformulations focused and consistent rather than unpredictably creative.
        temperature=0.3,
    )

    raw = (response.choices[0].message.content or "").strip()
    lines = [line.strip() for line in raw.splitlines() if line.strip()]

    reformulations = []
    for line in lines:
        # Strip common bullet prefixes in case the model ignores "no bullets"
        for prefix in ("- ", "* ", "  "):
            if line.startswith(prefix):
                line = line[len(prefix):].strip()
                break

        # Strip simple numeric prefixes in case the model uses numbering
        if len(line) >= 3 and line[0].isdigit() and line[1] in (".", ")") and line[2] == " ":
            line = line[3:].strip()

        lowered = line.lower()

        # Skip commentary lines if the model adds preamble
        if lowered.startswith(("here are", "alternative", "reformulation", "sure", "note:", "explanation:")):
            continue

        # Skip exact echoes of the original query
        if lowered == query.lower():
            continue

        if line:
            reformulations.append(line)
        if len(reformulations) >= MAX_REFORMULATIONS:
            break

    return reformulations if reformulations else [query]

In [ ]:
reformulations = expand_query("how do I unstage a file I accidentally added?")
for r in reformulations:
    print(r)

In [10]:
query = "how do I unstage a file I accidentally added?"

In [ ]:
EXPANSION_PROMPT.format(query=query)
# tried on local model

"You are helping improve search over Git documentation.\n\nGiven a user's question, generate 2 or 3 alternative phrasings that use different vocabulary but ask the same thing. Use terminology that might appear in official Git documentation (command names, flags, technical terms).\n\nReturn ONLY the alternative phrasings, one per line, with no numbering, no bullet points, no explanation, and no blank lines.\n\nUser question: how do I unstage a file I accidentally added?"

In [ ]:
def expand_and_retrieve(query, n_results=10):
    reformulations = expand_query(query)
    all_queries = [query] + reformulations
    seen = {}
    for q in all_queries:
        for chunk in retrieve(q, n_results=n_results):
            cid = chunk["chunk_id"]
            if cid not in seen or chunk["distance"] < seen[cid]["distance"]:
                seen[cid] = chunk
    return sorted(seen.values(), key=lambda x: x["distance"])

In [ ]:
results = expand_and_retrieve("how do I unstage a file I accidentally added?")
for i, chunk in enumerate(results, start=1):
    # The <-- GOLD marker uses the chunk_id previously inspected 
    marker = " <-- GOLD" if chunk["chunk_id"] == "d5237eadab2f87ed" else ""
    print(f"  [{i}] {chunk['distance']:.4f}  {chunk['title']}{marker}")

In [ ]:
"""
For a query like "how do I move commits from one branch to another?",
the preferred command is git cherry-pick, which applies specific commits
from one branch onto another. But a user who doesn't know that command
exists wouldn't naturally use the term "cherry-pick" in their question.

Multi-query expansion can only generate reformulations using vocabulary
already present in the query. It might produce alternatives like
"transfer commits between branches" or "relocate commits to a different branch,"
which are reasonable in plain English but don't appear in the cherry-pick
documentation. When the correct technical term is completely absent from the user's question,
expansion has little to work with.
"""

In [ ]:
# NOTE: HyDE (Hypothetical Document Embeddings.) and Choosing an Expansion Strategy

In [ ]:
from gitquest import corpus, collection

load_dotenv()

co = cohere.Client(api_key=os.getenv("COHERE_API_KEY"))
oai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

HYDE_PROMPT = """You are a technical writer for Git documentation.

Write a short documentation snippet (3-5 sentences) that directly answers
the following question. Write in the style of an official Git manpage or
user manual: technical, precise, using correct Git terminology and command
names. Include the relevant command syntax.

Question: {query}

Documentation snippet:"""

def generate_hypothetical_doc(query):
    response = oai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": HYDE_PROMPT.format(query=query)}
        ],
        temperature=0.3
    )
    return response.choices[0].message.content.strip()

def hyde_retrieve(query, n_results=20):
    hypothetical_doc = generate_hypothetical_doc(query)

    # Embed as a document, not a query, so it lands in the same
    # part of the vector space as the actual corpus chunks
    response = co.embed(
        texts=[hypothetical_doc],
        model="embed-v4.0",
        input_type="search_document",
        embedding_types=["float"]
    )
    embedding = response.embeddings.float[0]

    results = collection.query(
        query_embeddings=[embedding],
        n_results=n_results
    )

    chunks = []
    for chunk_id, metadata, distance in zip(
        results["ids"][0],
        results["metadatas"][0],
        results["distances"][0]
    ):
        chunk = corpus[chunk_id]
        chunks.append({
            "chunk_id": chunk_id,
            "text": chunk["text"],
            "distance": distance,
            "title": chunk["title"],
            "command": metadata["command"],
            "source_type": metadata["source_type"]
        })
    return hypothetical_doc, chunks

# Add your test code below this line
hypo_doc, results = hyde_retrieve("how do I move commits from one branch to another?")
print("Hypothetical document:")
print(hypo_doc)
print()
for i, chunk in enumerate(results[:5], start=1):
    marker = " <-- GOLD" if chunk["chunk_id"] == "bc750e0e88a5bb43" else ""
    print(f"  [{i}] {chunk['distance']:.4f}  {chunk['title']}{marker}")

In [ ]:
# no guarantee the HyDE approach will generate a hypothetical document that uses the right terminology
hypo_doc, results = hyde_retrieve("how do I unstage a file I accidentally added?")
print("Hypothetical document:")
print(hypo_doc)
print()
for i, chunk in enumerate(results[:5], start=1):
    marker = " <-- GOLD" if chunk["chunk_id"] == "d5237eadab2f87ed" else ""
    print(f"  [{i}] {chunk['distance']:.4f}  {chunk['title']}{marker}")

"""
The gold chunk doesn't appear anywhere in the top 20.
When asked to write a documentation snippet about unstaging,
the LLM reached straight for git reset, the same vocabulary pattern we've been seeing.
The hypothetical document landed near reset and "fixing mistakes" content,
pulling retrieval in the wrong direction. The vocabulary gap that defeated
multi-query expansion defeats HyDE here too, just via a different mechanism.

seen that HyDE isn't the right fit for GitQuest.
Its results are too dependent on whether the LLM happens
to reach for current terminology, which isn't something we can reasonably rely on.
"""


In [ ]:
"""
Multi-query expansion is the chosen expansion strategy,
and it does a good job of broadening the candidate pool.
What it doesn't do is guarantee that the most relevant chunks rise to the top of that pool.

recall and precision - talk about retrieval quality.

Recall asks whether the right chunk made it into our candidate pool at all.
Multi-query expansion improves recall by casting a wider net across multiple query phrasings.

Precision asks whether the most relevant chunks are ranked highly
enough to actually get sent to the LLM.
This is where embedding-based similarity search can be down.
It finds chunks whose vectors are geometrically close to the query vector,
which is a reasonable approximation for relevance, but not always an accurate one

A more precise approach is to read the query and each candidate document together
and score their relationship directly. A model that does this is called a cross-encoder.
It processes the query and document as a pair rather than as independent vectors.

The tradeoff for this strategy is speed.
Joint query-document scoring is more computationally expensive than vector comparison,
so it should be used as a focused second stage after embedding search has already narrowed the field.
"""

"\nMulti-query expansion is the chosen expansion strategy,\nand it does a good job of broadening the candidate pool.\nWhat it doesn't do is guarantee that the most relevant chunks rise to the top of that pool.\n\nrecall and precision - talk about retrieval quality.\n\nRecall asks whether the right chunk made it into our candidate pool at all.\nMulti-query expansion improves recall by casting a wider net across multiple query phrasings.\n\nPrecision asks whether the most relevant chunks are ranked highly\nenough to actually get sent to the LLM.\nThis is where embedding-based similarity search can be down.\n"

In [ ]:
def rerank(query, chunks, top_n=5):
    """
    
    - model="rerank-v3.5" is one of Cohere's reranking models.
      Unlike embedding models, reranking models don't produce vectors.
      They produce relevance scores, and rerank-v3.5 is optimized specifically for that task.
      NOTE: what is the cohere rerank API's relevance score depend on.
    
    - documents=documents passes the full text of each candidate chunk to the API.
      This is what enables the joint query-document scoring we discussed above.
      The model needs to read both to produce a meaningful relevance score.
    
    - result.index maps each result back to the original chunk in our list,
      so we can carry forward all the metadata we attached during retrieval.
    top_n=5 tells the reranker how many results to return. We're asking it to give us the 5 most relevant chunks from whatever candidate pool we pass in.
    """
    documents = [c["text"] for c in chunks]
    response = co.rerank(
        model="rerank-v3.5",
        query=query,
        documents=documents,
        top_n=top_n
    )
    reranked = []
    for result in response.results:
        chunk = chunks[result.index]
        reranked.append({
            **chunk,
            "rerank_score": result.relevance_score
        })
    return reranked

In [ ]:
# RERANK BEING USEFUL
query = "how do I create a new branch?"
candidates = expand_and_retrieve(query, n_results=10)
reranked = rerank(query, candidates, top_n=5)

print("Before reranking:")
for i, chunk in enumerate(candidates[:10], start=1):
    marker = " <-- GOLD" if chunk["chunk_id"] == "117d846e6e37c153" else ""
    print(f"  [{i}] {chunk['distance']:.4f}  {chunk['title']}{marker}")

print("\nAfter reranking:")
for i, chunk in enumerate(reranked, start=1):
    marker = " <-- GOLD" if chunk["chunk_id"] == "117d846e6e37c153" else ""
    print(f"  [{i}] {chunk['rerank_score']:.4f}  {chunk['title']}{marker}")

In [ ]:
# RERANK can FAIL!!!!!!!!
query = "how do I unstage a file I accidentally added?"
candidates = expand_and_retrieve(query, n_results=10)
reranked = rerank(query, candidates, top_n=5)

print("Before reranking (top 10):")
for i, chunk in enumerate(candidates[:10], start=1):
    marker = " <-- GOLD" if chunk["chunk_id"] == "d5237eadab2f87ed" else ""
    print(f"  [{i}] {chunk['distance']:.4f}  {chunk['title']}{marker}")

print("\nAfter reranking:")
for i, chunk in enumerate(reranked, start=1):
    marker = " <-- GOLD" if chunk["chunk_id"] == "d5237eadab2f87ed" else ""
    print(f"  [{i}] {chunk['rerank_score']:.4f}  {chunk['title']}{marker}")

In [ ]:
"""
NOTE: Context Window Management

...
There's also a subtler issue with LLMs in that they tend to pay less
attention to context placed in the middle of long prompts,
so padding out the context with lower-ranked chunks could
hurt answer quality even when the best chunk is present.
...
"""
# tiktoken is OpenAI's tokenizer library.
enc = tiktoken.encoding_for_model("gpt-4o-mini")

def count_tokens(text):
    return len(enc.encode(text))

query = "how do I see the commit history?"
base_prompt_tokens = count_tokens(SYSTEM_PROMPT.format(context=""))
query_tokens = count_tokens(query)
print(f"Base system prompt (no context): {base_prompt_tokens} tokens")
print(f"Query: {query_tokens} tokens")
print(f"Overhead before context: {base_prompt_tokens + query_tokens} tokens")
print()

for candidate_k in [3, 5, 10, 20]:
    chunks = expand_and_retrieve(query, n_results=candidate_k)
    context = build_context(chunks)
    total = base_prompt_tokens + query_tokens + count_tokens(context)
    pct = (total / 128000) * 100
    print(f"candidate_k={candidate_k:<3}  {total:>6,} tokens total  ({pct:.1f}% of 128k limit)")

In [ ]:
def select_chunks_within_budget(chunks, token_budget=6000):
    selected = []
    used = 0
    for chunk in chunks:
        chunk_tokens = count_tokens(build_context([chunk]))
        if used + chunk_tokens <= token_budget:
            selected.append(chunk)
            used += chunk_tokens
        else:
            continue
    return selected